# Calibrating an LLM Judge Against Human Labels

### An agreement score measures conformance to a policy — and if nobody wrote the policy down, it measures nothing

Most eval guides show you how to *build* an LLM judge. Very few show you how to find out whether your judge is any good, and almost none prepare you for what the first measurement actually reveals.

This notebook runs a controlled experiment on 45 customer-support examples:

- **Three judge prompts.** One leaves the grading policy unstated. One states a *permissive* policy (extra detail is fine). One states a *strict* policy (unverifiable additions are a failure).
- **Two sets of human labels** over the identical 45 examples — one produced under the strict policy, one under the permissive policy. They differ on 16 of 45 rows.
- **One 3×2 agreement matrix**, scored from a single cached set of judge verdicts.

The headline result:

| judge prompt | vs **strict** labels | vs **permissive** labels |
| --- | --- | --- |
| v1 — policy unstated | 55.6%  (κ 0.23) | 86.7%  (κ 0.68) |
| v2 — permissive stated | 60.0%  (κ 0.29) | **91.1%  (κ 0.79)** |
| v3 — strict stated | **84.4%  (κ 0.60)** | 48.9%  (κ 0.18) |

**The best prompt flips between the columns, and the same v3 prompt swings 35 points** — from best to worst — with no change to the judge, the model, the temperature, or the data. Only the labeller's rule changed.

> **The thesis:** a judge prompt does not encode skill, it encodes a policy. An agreement score therefore measures *distance from whichever policy someone happened to write down*. You can raise agreement 30 points either by improving the judge or by changing your mind about what "correct" means — and the number alone cannot tell those apart.

Everything here is standard library plus the `openai` SDK. No dataframe library, no plotting — the arithmetic is simple enough to read, which matters when the entire subject is trusting your measurements.

## Why this matters

A judge sits in a uniquely dangerous position: it is the thing that tells you whether everything *else* is working. When it diverges from what your team actually wants, every downstream signal diverges with it, silently, and the failure looks exactly like success — green dashboards, stable scores, and arguments about individual verdicts that never resolve.

| Failure | What you see | What is actually happening |
| --- | --- | --- |
| Judge is too lenient | Pass rates look healthy | Real regressions ship |
| Judge is too strict | Constant red builds | Teams stop trusting the gate and route around it |
| Judge encodes an **unstated policy** | Scores look stable; verdict disputes never converge | Nobody agreed what "correct" means, so every dispute is unwinnable |

The third is what this notebook is about, and it is the hardest to see, because it does not look like a bug. It looks like people disagreeing about examples.

None of this needs many labels. Forty-five carefully labelled examples told us more about our own definitions than thousands of unlabelled generations would have.

In [1]:
%pip install --quiet openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import os
from collections import Counter
from pathlib import Path
from statistics import median

from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from the environment

# The model acting as judge. Set JUDGE_MODEL in your environment to override.
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "gpt-4.1-mini")

print(f"judge model: {JUDGE_MODEL}")

judge model: gpt-4.1-mini


## The dataset

A small customer-support QA set: each row is a question, a reference answer, and a candidate answer from some system under test.

Two properties are deliberate:

- **Answer length varies widely** (2 to 60 words). Length is computable without a label, which makes it a usable slicing axis later.
- **Many candidates add detail the reference never mentions.** Those rows are where the policy question bites, and they are the ones the two labelling passes disagree about.

### The choice nobody makes explicitly

A **reference answer** is not a **source document**.

| | Source document | Reference answer |
| --- | --- | --- |
| What it is | Ground truth about the world | One acceptable answer |
| Question it supports | Is this claim supported by the source? | Does the candidate assert the required facts? |
| Metric | Groundedness / faithfulness | Correctness / equivalence |

A reference states what a correct answer **must contain**. On its own it says nothing about what an answer is **permitted** to contain — and that silence is where the trouble lives.

Take a candidate that gets the required fact right and adds three claims the reference never makes. Pass or fail?

- **Permissive policy:** pass. The required fact is present; extra detail helps the user.
- **Strict policy:** fail. The extra claims cannot be checked against anything, so they are unaudited assertions being shown to a customer.

**Both are defensible and they are opposite.** A regulated support product might reasonably choose strict; a developer-docs assistant might reasonably choose permissive. What is *not* defensible is failing to choose — because then your judge picks silently, your humans pick differently, and the resulting disagreement looks like a quality problem when it is a specification problem.

This notebook labels the same data **both ways** so you can see exactly how much of a judge's apparent quality is really just policy alignment.

In [3]:
# The dataset is inlined so this notebook is self-contained and runnable as a single file.
# Swap it for your own rows: id, question, reference_answer, candidate_answer.
EXAMPLES = [
    {
        "id": 'q01',
        "question": 'What is the default session timeout?',
        "reference_answer": 'The default session timeout is 30 minutes of inactivity.',
        "candidate_answer": '30 minutes.',
    },
    {
        "id": 'q02',
        "question": 'How do I reset my password?',
        "reference_answer": 'Use the Forgot Password link on the login page to receive a reset email.',
        "candidate_answer": "Click Forgot Password on the login screen and we'll email you a reset link.",
    },
    {
        "id": 'q03',
        "question": 'Which plans include SSO?',
        "reference_answer": 'SSO is included in the Enterprise plan only.',
        "candidate_answer": 'SSO is available on some of our plans.',
    },
    {
        "id": 'q04',
        "question": 'What is the API rate limit?',
        "reference_answer": 'The API rate limit is 100 requests per minute per API key.',
        "candidate_answer": "You get 100 requests per minute for each key. If you exceed it you'll receive a 429 response, and the Retry-After header tells you how long to wait before retrying. Rate limits are applied per key rather than per account, so separate keys have separate budgets.",
    },
    {
        "id": 'q05',
        "question": 'Where is customer data stored?',
        "reference_answer": 'Customer data is stored in AWS us-east-1 with cross-region backups.',
        "candidate_answer": 'All customer data lives in AWS us-east-1. We also maintain cross-region backups for disaster recovery purposes, and backups are encrypted at rest using AES-256. If you have data residency requirements outside the US, contact your account team, since regional hosting is available on Enterprise agreements.',
    },
    {
        "id": 'q06',
        "question": 'Can I export my data?',
        "reference_answer": 'Yes, you can export all data as CSV from the settings page.',
        "candidate_answer": 'Yes — go to Settings and choose Export. You can download everything as CSV.',
    },
    {
        "id": 'q07',
        "question": 'How long are backups retained?',
        "reference_answer": 'Backups are retained for 30 days.',
        "candidate_answer": 'Backups are retained for 90 days.',
    },
    {
        "id": 'q08',
        "question": 'Do you support two-factor authentication?',
        "reference_answer": 'Yes, TOTP-based two-factor authentication is supported for all plans.',
        "candidate_answer": 'Yes, we support two-factor authentication using authenticator apps such as Google Authenticator or Authy, on every plan including the free tier. You enable it from your profile security settings, and we recommend saving the recovery codes somewhere safe because losing your device without them means contacting support to regain access.',
    },
    {
        "id": 'q09',
        "question": 'What happens when my trial ends?',
        "reference_answer": 'At the end of the trial your account moves to the free tier and premium features are disabled.',
        "candidate_answer": 'Your account downgrades to the free plan and you lose access to premium features.',
    },
    {
        "id": 'q10',
        "question": 'Is there a mobile app?',
        "reference_answer": 'Yes, native apps are available for iOS and Android.',
        "candidate_answer": 'Yes, we have mobile apps.',
    },
    {
        "id": 'q11',
        "question": 'How do I invite a teammate?',
        "reference_answer": 'Go to Settings > Members and enter their email address to send an invitation.',
        "candidate_answer": "Open Settings, then Members, then type in their email and hit Invite. They'll get an email with a join link that expires after seven days. If they don't receive it, ask them to check spam, and you can resend the invitation from the same screen. Admins can also set the teammate's role at invite time rather than changing it afterwards.",
    },
    {
        "id": 'q12',
        "question": 'What is the maximum file upload size?',
        "reference_answer": 'The maximum file upload size is 25 MB.',
        "candidate_answer": 'Files can be up to 25 MB.',
    },
    {
        "id": 'q13',
        "question": 'Do you offer a student discount?',
        "reference_answer": 'Yes, students receive 50% off with a valid .edu email address.',
        "candidate_answer": 'Yes, there is a discount for students.',
    },
    {
        "id": 'q14',
        "question": 'Which browsers are supported?',
        "reference_answer": 'Chrome, Firefox, Safari and Edge are supported on their latest two versions.',
        "candidate_answer": "We support Chrome, Firefox, Safari and Edge. Specifically, we test against the latest two major versions of each, so if you're on something older you may hit rendering issues. Internet Explorer is not supported and hasn't been since 2022.",
    },
    {
        "id": 'q15',
        "question": 'How do I cancel my subscription?',
        "reference_answer": 'Cancel from Settings > Billing; access continues until the end of the current billing period.',
        "candidate_answer": "You can cancel anytime under Settings then Billing. Your plan stays active until the end of the period you've already paid for, so you won't lose access immediately, and you won't be charged again after that. We don't offer prorated refunds for partial months.",
    },
    {
        "id": 'q16',
        "question": 'Is my data encrypted in transit?',
        "reference_answer": 'Yes, all traffic uses TLS 1.2 or higher.',
        "candidate_answer": 'Yes, everything is encrypted with TLS.',
    },
    {
        "id": 'q17',
        "question": 'Can I use my own domain?',
        "reference_answer": 'Custom domains are available on Business and Enterprise plans.',
        "candidate_answer": 'Yes, custom domains are supported on all plans.',
    },
    {
        "id": 'q18',
        "question": 'How many API keys can I create?',
        "reference_answer": 'You can create up to 10 active API keys per account.',
        "candidate_answer": "Up to 10 active keys per account. Revoked keys don't count toward the limit, so if you're rotating keys you can revoke the old one first and stay within the cap. Keys are shown once at creation time and can't be retrieved later.",
    },
    {
        "id": 'q19',
        "question": 'What time zone are reports generated in?',
        "reference_answer": 'Reports are generated in UTC.',
        "candidate_answer": 'Reports use UTC.',
    },
    {
        "id": 'q20',
        "question": 'Do you have a status page?',
        "reference_answer": 'Yes, service status is published at status.example.com.',
        "candidate_answer": "Yes, you can check status.example.com for live incident and uptime information. We also post updates there during planned maintenance windows, and you can subscribe to receive email or SMS alerts whenever the status changes so you don't have to poll it manually.",
    },
    {
        "id": 'q21',
        "question": 'How do I contact support?',
        "reference_answer": 'Support is available via the in-app chat widget or by emailing support@example.com.',
        "candidate_answer": 'Email support@example.com.',
    },
    {
        "id": 'q22',
        "question": 'Is there an SLA?',
        "reference_answer": 'Enterprise customers receive a 99.9% uptime SLA.',
        "candidate_answer": "We offer a 99.9% uptime SLA to Enterprise customers. It covers the core API and web application, with service credits issued if we fall below the threshold in a given month. Credits have to be requested within 30 days of the incident, and they're applied to your next invoice rather than refunded.",
    },
    {
        "id": 'q23',
        "question": 'Can I change my plan mid-cycle?',
        "reference_answer": 'Yes, upgrades take effect immediately and downgrades apply at the next billing cycle.',
        "candidate_answer": 'Yes, you can change plans whenever you like.',
    },
    {
        "id": 'q24',
        "question": 'What payment methods do you accept?',
        "reference_answer": 'We accept major credit cards and, for annual Enterprise contracts, bank transfer.',
        "candidate_answer": "Credit cards are accepted, and Enterprise customers on annual contracts can pay by bank transfer instead. We don't currently support PayPal or cryptocurrency. Invoices are issued at the start of each billing period and payment terms are net 30 for contracted customers.",
    },
    {
        "id": 'q25',
        "question": 'How often is the search index updated?',
        "reference_answer": 'The search index updates every 15 minutes.',
        "candidate_answer": "The index refreshes every 15 minutes, so newly created items generally become searchable within that window. Bulk imports can take longer because they're queued behind regular traffic, and very large imports may take up to an hour to appear.",
    },
    {
        "id": 'q26',
        "question": 'Do you delete data when I close my account?',
        "reference_answer": 'Data is permanently deleted 30 days after account closure.',
        "candidate_answer": 'Yes, your data is deleted immediately when you close the account.',
    },
    {
        "id": 'q27',
        "question": 'Can I restrict access by IP address?',
        "reference_answer": 'IP allowlisting is available on Enterprise plans.',
        "candidate_answer": 'IP allowlisting is an Enterprise feature. You configure it under Settings then Security, and you can specify individual addresses or CIDR ranges. Be careful adding your own address first, because locking yourself out requires a support ticket to resolve.',
    },
    {
        "id": 'q28',
        "question": 'Is there a free tier?',
        "reference_answer": 'Yes, the free tier includes up to 3 users and 1 GB of storage.',
        "candidate_answer": "Yes, there's a free tier available.",
    },
    {
        "id": 'q29',
        "question": 'How do webhooks handle retries?',
        "reference_answer": 'Failed webhooks are retried up to 5 times with exponential backoff.',
        "candidate_answer": "If your endpoint doesn't return a 2xx, we retry up to five times using exponential backoff, spacing the attempts further apart each time. After the fifth failure the delivery is marked failed and you can replay it manually from the dashboard. We recommend responding quickly and processing asynchronously, since we time out after 10 seconds.",
    },
    {
        "id": 'q30',
        "question": 'What is the audit log retention period?',
        "reference_answer": 'Audit logs are retained for 1 year on Enterprise plans.',
        "candidate_answer": 'Audit logs are kept for one year.',
    },
    {
        "id": 'q31',
        "question": 'Can I set up single sign-on with Okta?',
        "reference_answer": 'Yes, SAML-based SSO works with Okta and other identity providers on Enterprise plans.',
        "candidate_answer": "Yes, we support SAML SSO and Okta is one of the identity providers customers use most. Setup involves exchanging metadata between our app and your Okta tenant, and it's available on Enterprise. Azure AD, Google Workspace and OneLogin work the same way through the generic SAML connector.",
    },
    {
        "id": 'q32',
        "question": 'Does the API support pagination?',
        "reference_answer": 'Yes, list endpoints use cursor-based pagination with a default page size of 50.',
        "candidate_answer": 'Yes, the API paginates results.',
    },
    {
        "id": 'q33',
        "question": 'How do I rotate an API key?',
        "reference_answer": 'Create a new key in Settings > API, update your integration, then revoke the old key.',
        "candidate_answer": "Generate a replacement key from the API settings page, switch your integration over to it, and once you've confirmed traffic is flowing on the new key, revoke the old one. Doing it in that order avoids downtime, since both keys are valid during the overlap.",
    },
    {
        "id": 'q34',
        "question": 'Are there usage analytics?',
        "reference_answer": 'Yes, usage analytics are available in the dashboard for the last 90 days.',
        "candidate_answer": 'Yes, the dashboard shows usage analytics covering the previous 90 days.',
    },
    {
        "id": 'q35',
        "question": 'Can multiple people edit at the same time?',
        "reference_answer": 'Yes, real-time collaborative editing is supported.',
        "candidate_answer": "Yes, multiple people can edit simultaneously and you'll see each other's cursors and changes as they happen. Edits are merged automatically, and there's a version history you can roll back to if something goes wrong. Presence indicators show who else currently has the document open.",
    },
    {
        "id": 'q36',
        "question": 'What is the notification delivery latency?',
        "reference_answer": 'Notifications are typically delivered within 60 seconds.',
        "candidate_answer": 'Notifications usually arrive within a minute.',
    },
    {
        "id": 'q37',
        "question": 'Do you support GDPR data subject requests?',
        "reference_answer": 'Yes, data subject access and deletion requests are handled within 30 days.',
        "candidate_answer": "Yes, we handle GDPR data subject requests including access and deletion, and we respond within the 30-day statutory window. Requests come in through your account team or the privacy email address, and we'll confirm identity before acting on a deletion since those can't be undone.",
    },
    {
        "id": 'q38',
        "question": 'Is there a sandbox environment?',
        "reference_answer": 'Yes, a sandbox environment is available with separate API keys.',
        "candidate_answer": "Yes, there's a sandbox.",
    },
    {
        "id": 'q39',
        "question": 'How are seats counted for billing?',
        "reference_answer": 'Billing counts active users at the end of each billing period.',
        "candidate_answer": 'Seats are counted at the start of the billing period.',
    },
    {
        "id": 'q40',
        "question": 'Can I schedule reports?',
        "reference_answer": 'Yes, reports can be scheduled daily, weekly or monthly and emailed to recipients.',
        "candidate_answer": "Yes, you can schedule a report to run daily, weekly or monthly, and have it emailed to whoever needs it. Recipients don't need an account to receive the emailed copy, though they will need one to open the interactive version in the dashboard.",
    },
    {
        "id": 'q41',
        "question": 'What happens if I exceed my storage limit?',
        "reference_answer": 'Uploads are blocked once the storage limit is reached until space is freed or the plan is upgraded.',
        "candidate_answer": "You'll be charged overage fees automatically.",
    },
    {
        "id": 'q42',
        "question": 'Does the platform have an audit trail for admin actions?',
        "reference_answer": 'Yes, all administrative actions are recorded in the audit log.',
        "candidate_answer": "Yes, admin actions are captured in the audit log, including role changes, member removals, security setting updates and API key creation or revocation. Each entry records the actor, the action, the timestamp and the source IP address, and the log is append-only so entries can't be edited or deleted after the fact.",
    },
    {
        "id": 'q43',
        "question": 'Can I use the API without an account?',
        "reference_answer": 'No, an account and API key are required to use the API.',
        "candidate_answer": 'No — you need an account and an API key.',
    },
    {
        "id": 'q44',
        "question": 'How do I transfer ownership of the account?',
        "reference_answer": 'Account ownership transfer is performed by support after verifying both parties.',
        "candidate_answer": 'You can transfer ownership yourself from the Members page.',
    },
    {
        "id": 'q45',
        "question": 'Are there webhooks for billing events?',
        "reference_answer": 'Yes, webhooks are available for subscription and invoice events.',
        "candidate_answer": "Yes, you can subscribe to webhooks for billing events such as subscription created, updated and cancelled, plus invoice paid and payment failed. They use the same delivery and retry mechanism as other webhooks, and the payload includes the affected subscription or invoice object so you usually don't need a follow-up API call.",
    },
]

examples = [dict(r) for r in EXAMPLES]
for row in examples:
    row["candidate_words"] = len(row["candidate_answer"].split())

lengths = sorted(r["candidate_words"] for r in examples)
print(f"{len(examples)} examples")
print(f"candidate length: median {median(lengths):.0f} words, range {lengths[0]}-{lengths[-1]}")
print()

# A terse candidate and one that adds unrequested detail - the two shapes that matter.
for row in (examples[0], examples[3]):
    print(f"[{row['id']}] {row['question']}")
    print(f"  reference ({len(row['reference_answer'].split())}w): {row['reference_answer']}")
    print(f"  candidate ({row['candidate_words']}w): {row['candidate_answer']}")
    print()

45 examples
candidate length: median 14 words, range 2-60

[q01] What is the default session timeout?
  reference (9w): The default session timeout is 30 minutes of inactivity.
  candidate (2w): 30 minutes.

[q04] What is the API rate limit?
  reference (12w): The API rate limit is 100 requests per minute per API key.
  candidate (46w): You get 100 requests per minute for each key. If you exceed it you'll receive a 429 response, and the Retry-After header tells you how long to wait before retrying. Rate limits are applied per key rather than per account, so separate keys have separate budgets.



## Step 1 — Three judges, three policies

**v1** looks like the prompt most teams actually ship. Read it and notice what it does *not* say: nothing about extra detail, nothing about unverifiable additions, nothing about length. It states no policy — so the model will infer one, consistently and invisibly.

**v2** states the permissive policy explicitly.

**v3** states the strict policy explicitly.

None of the three is more sophisticated than the others. They differ only in which rule they encode.

In [4]:
JUDGE_PROMPT_V1_UNSTATED = """You are grading answers from a customer support assistant.

Compare the CANDIDATE answer to the REFERENCE answer.
Return PASS if the candidate answer is correct, FAIL if it is not.

QUESTION: {question}
REFERENCE: {reference_answer}
CANDIDATE: {candidate_answer}
"""

JUDGE_PROMPT_V2_PERMISSIVE = """You are grading answers from a customer support assistant.

Decide whether the CANDIDATE answer is factually consistent with the REFERENCE answer.

Grade PASS if every factual claim required by the reference is present and correct in the candidate.
Grade FAIL only if the candidate contradicts the reference or omits a required fact.

Important:
- The reference states the required facts. It is NOT a target style, length, or wording.
- Additional correct detail, context, or caveats are NOT a reason to fail an answer.
- A longer answer is not a worse answer. Judge only factual agreement.

QUESTION: {question}
REFERENCE: {reference_answer}
CANDIDATE: {candidate_answer}
"""

JUDGE_PROMPT_V3_STRICT = """You are grading answers from a customer support assistant whose answers
are shown directly to customers.

Grade the CANDIDATE against the REFERENCE using exactly these rules:

PASS only if ALL of the following hold:
  1. Every fact asserted by the reference is present and correct in the candidate.
  2. The candidate does not contradict the reference.
  3. The candidate does not add factual claims that cannot be verified against the reference.

Otherwise FAIL, and say which of the three rules was broken.

Rule 3 is deliberate. This assistant's answers reach customers, and an unverifiable claim is a
liability whether or not it happens to be true. Extra detail that merely restates or clarifies the
reference is acceptable; extra detail that introduces NEW facts not present in the reference is not.

QUESTION: {question}
REFERENCE: {reference_answer}
CANDIDATE: {candidate_answer}
"""

PROMPTS = {
    "v1_unstated": JUDGE_PROMPT_V1_UNSTATED,
    "v2_permissive": JUDGE_PROMPT_V2_PERMISSIVE,
    "v3_strict": JUDGE_PROMPT_V3_STRICT,
}

VERDICT_FORMAT = {
    "format": {
        "type": "json_schema",
        "name": "verdict",
        "schema": {
            "type": "object",
            "properties": {
                "verdict": {"type": "string", "enum": ["PASS", "FAIL"]},
                "reason": {"type": "string"},
            },
            "required": ["verdict", "reason"],
            "additionalProperties": False,
        },
        "strict": True,
    }
}


def judge_one(row: dict, prompt_template: str) -> dict:
    """Score one example. Returns {'verdict': 'PASS'|'FAIL', 'reason': str}."""
    prompt = prompt_template.format(
        question=row["question"],
        reference_answer=row["reference_answer"],
        candidate_answer=row["candidate_answer"],
    )
    response = client.responses.create(
        model=JUDGE_MODEL,
        input=prompt,
        temperature=0,          # drop this if your model rejects it; note the loss of determinism
        text=VERDICT_FORMAT,
    )
    return json.loads(response.output_text)

### Grade once, score twice

**A judge's verdicts do not depend on the labels.** That sounds obvious and it has a useful consequence: one grading pass serves every label set you will ever compare against, so adding a third labelling policy later costs nothing.

Three prompts over 45 examples is 135 calls.

In [5]:
# A judge's verdicts do not depend on the labels, so one grading pass serves every
# label set you compare against. 3 prompts x 45 examples = 135 calls.
verdicts: dict[str, dict[str, str]] = {}

for name, template in PROMPTS.items():
    print(f"grading with {name} ...")
    for i, row in enumerate(examples, 1):
        result = judge_one(row, template)
        entry = verdicts.setdefault(row["id"], {})
        entry[name] = result["verdict"]
        entry[f"{name}_reason"] = result["reason"]
        if i % 15 == 0:
            print(f"  {i}/{len(examples)}")

for row in examples:
    row.update(verdicts[row["id"]])

print()
for name in PROMPTS:
    print(f"{name:>15}: {dict(Counter(r[name] for r in examples))}")

grading with v1_unstated ...
  15/45
  30/45
  45/45
grading with v2_permissive ...
  15/45
  30/45
  45/45
grading with v3_strict ...
  15/45
  30/45
  45/45

    v1_unstated: {'PASS': 33, 'FAIL': 12}
  v2_permissive: {'PASS': 31, 'FAIL': 14}
      v3_strict: {'FAIL': 37, 'PASS': 8}


## Step 2 — Two labellers, two policies, the same 45 examples

**Write the labelling rule before labelling anything.** Without a written rule your own labels drift, and you end up measuring your inconsistency instead of the judge's. The rule is also the artifact you will compare against the judge's *implicit* rule, so it has to exist in writing.

We labelled the same data twice, under two rules that differ in exactly one clause:

**Strict policy** (`human_labels_strict.csv`)
> PASS if every fact the reference asserts is present and correct.
> FAIL on contradiction. FAIL on a missing required fact.
> **FAIL if the candidate adds claims that cannot be verified against the reference.**

**Permissive policy** (`human_labels_permissive.csv`)
> PASS if every fact the reference asserts is present and correct.
> FAIL on contradiction. FAIL on a missing required fact.
> **Extra correct detail is not a defect.**

That single clause moved **16 of 45 labels**, all in the same direction. Two rules that a reasonable team could argue for, and a third of the dataset changes verdict.

The loader below sniffs the delimiter and strips a byte-order mark, because spreadsheet exports vary — Excel writes comma-with-BOM as "CSV" and tab-separated as "Text", and a labelling file that fails to parse is how a calibration exercise dies on the first afternoon.

In [6]:
# Two human passes over the same 45 examples, differing only in how they treat
# unverifiable extra detail. Inlined here; in practice these come from a
# spreadsheet export - see the note below on parsing those robustly.
STRICT_LABELS = {"q01": "PASS", "q02": "PASS", "q03": "FAIL", "q04": "FAIL", "q05": "FAIL", 
    "q06": "PASS", "q07": "FAIL", "q08": "FAIL", "q09": "PASS", "q10": "FAIL", "q11": "FAIL", 
    "q12": "PASS", "q13": "FAIL", "q14": "PASS", "q15": "FAIL", "q16": "FAIL", "q17": "FAIL", 
    "q18": "FAIL", "q19": "PASS", "q20": "FAIL", "q21": "FAIL", "q22": "FAIL", "q23": "FAIL", 
    "q24": "FAIL", "q25": "FAIL", "q26": "FAIL", "q27": "FAIL", "q28": "FAIL", "q29": "FAIL", 
    "q30": "FAIL", "q31": "PASS", "q32": "PASS", "q33": "PASS", "q34": "PASS", "q35": "FAIL", 
    "q36": "PASS", "q37": "FAIL", "q38": "FAIL", "q39": "FAIL", "q40": "PASS", "q41": "FAIL", 
    "q42": "PASS", "q43": "PASS", "q44": "FAIL", "q45": "FAIL"}

PERMISSIVE_LABELS = {"q01": "PASS", "q02": "PASS", "q03": "FAIL", "q04": "PASS", "q05": "PASS", 
    "q06": "PASS", "q07": "FAIL", "q08": "PASS", "q09": "PASS", "q10": "FAIL", "q11": "PASS", 
    "q12": "PASS", "q13": "FAIL", "q14": "PASS", "q15": "PASS", "q16": "FAIL", "q17": "FAIL", 
    "q18": "PASS", "q19": "PASS", "q20": "PASS", "q21": "FAIL", "q22": "PASS", "q23": "FAIL", 
    "q24": "PASS", "q25": "PASS", "q26": "FAIL", "q27": "PASS", "q28": "PASS", "q29": "PASS", 
    "q30": "FAIL", "q31": "PASS", "q32": "PASS", "q33": "PASS", "q34": "PASS", "q35": "PASS", 
    "q36": "PASS", "q37": "PASS", "q38": "FAIL", "q39": "FAIL", "q40": "PASS", "q41": "FAIL", 
    "q42": "PASS", "q43": "PASS", "q44": "FAIL", "q45": "PASS"}

POLICIES = {"strict": STRICT_LABELS, "permissive": PERMISSIVE_LABELS}

for name, labels in POLICIES.items():
    counts = Counter(labels.values())
    print(f"{name:>11}: {dict(counts)}  fail rate {counts['FAIL'] / len(labels):.0%}")

flipped = [k for k in STRICT_LABELS if STRICT_LABELS[k] != PERMISSIVE_LABELS[k]]
print(f"\n{len(flipped)} of {len(examples)} labels differ between the two policies:")
print("  " + " ".join(sorted(flipped)))

     strict: {'PASS': 15, 'FAIL': 30}  fail rate 67%
 permissive: {'PASS': 31, 'FAIL': 14}  fail rate 31%

16 of 45 labels differ between the two policies:
  q04 q05 q08 q11 q15 q18 q20 q22 q24 q25 q27 q28 q29 q35 q37 q45


### Loading labels from a spreadsheet

The labels above are inlined to keep this notebook self-contained. When you run this on your own
data they will come out of a spreadsheet, and that is where a calibration exercise most often dies
on its first afternoon — Excel writes "CSV" as comma-with-BOM and "Text" as tab-separated, and a
hard-coded `csv.DictReader(open(path))` fails on both in different ways.

```python
def read_label_file(path: Path) -> dict[str, str]:
    text = path.read_text(encoding="utf-8-sig")      # strips Excel's byte-order mark
    lines = text.splitlines()
    dialect = csv.Sniffer().sniff(lines[0], delimiters=",;\t")
    rows = list(csv.DictReader(lines, dialect=dialect))

    labels = {}
    for row in rows:
        verdict = (row.get("human") or "").strip().upper()
        assert verdict in {"PASS", "FAIL"}, f"{path.name} row {row.get('id')}: got {verdict!r}"
        labels[row["id"]] = verdict
    return labels
```

Sniff the delimiter, strip the BOM, and assert the verdict vocabulary — a silently mis-parsed label
file produces an agreement number that looks plausible and means nothing.

## Step 3 — Measure agreement

Raw agreement — what fraction of examples did judge and human label the same way — is worth computing and worth distrusting.

If 85% of your examples are passes, a judge that blindly answers PASS every time scores 85% agreement while containing no information at all. **Cohen's κ** corrects for that by asking how much better than chance the agreement is:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is observed agreement and $p_e$ is what you would expect if both raters guessed at their own marginal rates. κ = 1 is perfect, κ = 0 is chance, κ < 0 means flipping the judge's answer would do better.

**Also compute the trivial baseline** — the score a constant "always FAIL" predictor would get. A judge that loses to a constant has its threshold in the wrong place, and you cannot see that from agreement alone.

In [7]:
def cohens_kappa(a: list[str], b: list[str]) -> float:
    """Cohen's kappa for two equal-length lists of categorical labels."""
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    count_a, count_b = Counter(a), Counter(b)
    expected = sum((count_a[k] / n) * (count_b[k] / n) for k in set(a) | set(b))
    return 1.0 if expected >= 1 else (observed - expected) / (1 - expected)


def score(rows: list[dict], judge_key: str, labels: dict[str, str]) -> dict:
    human = [labels[r["id"]] for r in rows]
    verdicts = [r[judge_key] for r in rows]
    n = len(rows)
    return {
        "n": n,
        "raw": sum(h == j for h, j in zip(human, verdicts)) / n,
        "kappa": cohens_kappa(human, verdicts),
        "too_strict": sum(h == "PASS" and j == "FAIL" for h, j in zip(human, verdicts)),
        "too_lenient": sum(h == "FAIL" and j == "PASS" for h, j in zip(human, verdicts)),
        "baseline": max(Counter(human).values()) / n,  # best constant predictor
    }

In [8]:
# THE MATRIX: three judge prompts x two labelling policies, one dataset.
print(f"{'':>15}  {'vs STRICT labels':>26}   {'vs PERMISSIVE labels':>26}")
print(f"{'judge prompt':>15}  {'agree':>7} {'kappa':>7} {'lenient':>8}   {'agree':>7} {'kappa':>7} {'lenient':>8}")
print("-" * 82)

results = {}
for judge_key in PROMPTS:
    cells = []
    for policy, labels in POLICIES.items():
        s = score(examples, judge_key, labels)
        results[(judge_key, policy)] = s
        cells.append(f"{s['raw']:>7.1%} {s['kappa']:>7.2f} {s['too_lenient']:>8}")
    print(f"{judge_key:>15}  {cells[0]}   {cells[1]}")

print("-" * 82)
for policy, labels in POLICIES.items():
    base = max(Counter(labels.values()).values()) / len(labels)
    best = max(PROMPTS, key=lambda k: results[(k, policy)]["raw"])
    print(f"{policy:>11} labels: constant-predictor baseline {base:.1%}   best prompt: {best}")

                           vs STRICT labels         vs PERMISSIVE labels
   judge prompt    agree   kappa  lenient     agree   kappa  lenient
----------------------------------------------------------------------------------
    v1_unstated    55.6%    0.23       19     86.7%    0.68        4
  v2_permissive    60.0%    0.29       17     91.1%    0.79        2
      v3_strict    84.4%    0.60        0     48.9%    0.18        0
----------------------------------------------------------------------------------
     strict labels: constant-predictor baseline 66.7%   best prompt: v3_strict
 permissive labels: constant-predictor baseline 68.9%   best prompt: v2_permissive


### Reading the matrix

```
                    vs STRICT labels          vs PERMISSIVE labels
prompt              agree    kappa            agree    kappa
v1_unstated         55.6%     0.23            86.7%     0.68
v2_permissive       60.0%     0.29            91.1%     0.79
v3_strict           84.4%     0.60            48.9%     0.18

constant baseline   66.7%                     68.9%
best prompt         v3_strict                 v2_permissive
```

**1. The winner flips.** v3 is the best prompt against strict labels and the *worst* against permissive ones. Nothing about the judge changed between those two columns — the verdicts were graded once and cached, then scored twice. The only difference is the rule the labeller applied.

**2. The swing is enormous.** The same v3 prompt: **84.4% → 48.9%**, a 35.5-point drop, κ 0.60 → 0.18. In the other direction, v1 goes **55.6% → 86.7%**, κ 0.23 → 0.68.

**3. The same judge is "broken" or "good" depending only on the policy.** Compare v1 against the constant-predictor baseline in each column. Under strict labels it scores 55.6% against a 66.7% baseline — *beaten by a predictor that always answers FAIL*. Under permissive labels the identical verdicts score 86.7% against a 68.9% baseline — comfortably useful. Same model, same prompt, same outputs, opposite conclusions.

**4. The long-answer slice is a near-perfect policy detector.** From the slice table above:

| on long answers | vs strict | vs permissive |
| --- | --- | --- |
| v1 / v2 (permissive-leaning) | 26.7% | **100.0%** |
| v3 (strict) | 73.3% | **0.0%** |

Fifteen examples, and the two policies produce exact mirror images — 100% versus 0%. That is not a judge being unreliable; it is two rules disagreeing about the same fifteen answers, perfectly consistently.

**5. The error direction is one-sided and stable.** v3 is never more lenient than either labeller (0 in both columns); against permissive labels it is *stricter* 23 times. v1 and v2 are the reverse: 19 and 17 cases of being more lenient than the strict labeller. Each prompt is a consistent instrument. It is just an instrument calibrated to a particular rule.

The uncomfortable implication:

> **You can raise agreement by thirty points either by improving the judge or by changing your mind about what "correct" means. The agreement number cannot distinguish those two, and only one of them is engineering.**

This is why a rising eval score is not self-evidently good news, and why *"our judge agrees with humans 90% of the time"* is an incomplete sentence. Agreement with **whom**, grading under **which rule**?

## Step 4 — Where the disagreement lives

An aggregate number tells you *that* a judge diverges, never *where*. So slice by any property computable without a label — length, category, source system, whether the answer contains a number.

We slice by length, not because length causes anything, but because in this dataset it correlates with the thing that actually matters: whether the candidate added claims the reference never made.

In [9]:
def add_tertile_buckets(rows: list[dict], field: str = "candidate_words") -> None:
    ordered = sorted(rows, key=lambda r: r[field])
    n = len(ordered)
    for i, row in enumerate(ordered):
        row["length_bucket"] = "short" if i < n / 3 else ("medium" if i < 2 * n / 3 else "long")


add_tertile_buckets(examples)

for policy in POLICIES:
    print(f"--- agreement vs {policy} labels ---")
    print(f"{'bucket':>7}  " + "  ".join(f"{k:>13}" for k in PROMPTS))
    for bucket in ("short", "medium", "long"):
        subset = [r for r in examples if r["length_bucket"] == bucket]
        cells = [f"{score(subset, k, POLICIES[policy])['raw']:>12.1%}" for k in PROMPTS]
        print(f"{bucket:>7}  " + "   ".join(cells))
    print()

--- agreement vs strict labels ---
 bucket    v1_unstated  v2_permissive      v3_strict
  short         66.7%          80.0%          86.7%
 medium         73.3%          73.3%          93.3%
   long         26.7%          26.7%          73.3%

--- agreement vs permissive labels ---
 bucket    v1_unstated  v2_permissive      v3_strict
  short         60.0%          73.3%          80.0%
 medium        100.0%         100.0%          66.7%
   long        100.0%         100.0%           0.0%



In [10]:
# The rows where the two policies disagree — read what each side actually said.
print(f"{len(flipped)} rows labelled differently under the two policies.\n")

for row in [r for r in examples if r["id"] in flipped][:4]:
    print(f"[{row['id']}] {row['candidate_words']}w")
    print(f"   strict={POLICIES['strict'][row['id']]}  permissive={POLICIES['permissive'][row['id']]}")
    print(f"   v1 said {row['v1_unstated']}: {row['v1_unstated_reason'][:150]}")
    print(f"   v3 said {row['v3_strict']}: {row['v3_strict_reason'][:150]}")
    print()

16 rows labelled differently under the two policies.

[q04] 46w
   strict=FAIL  permissive=PASS
   v1 said PASS: The candidate answer correctly states the API rate limit as 100 requests per minute per key, matching the reference. Additionally, it provides extra u
   v3 said FAIL: The candidate adds factual claims not present in the reference, such as receiving a 429 response, the Retry-After header, and that rate limits are app

[q05] 45w
   strict=FAIL  permissive=PASS
   v1 said PASS: The candidate answer correctly states that customer data is stored in AWS us-east-1 and mentions cross-region backups, aligning with the reference ans
   v3 said FAIL: Rule 3 broken: The candidate adds unverifiable claims not present in the reference, such as backups being encrypted at rest using AES-256 and the avai

[q08] 50w
   strict=FAIL  permissive=PASS
   v1 said PASS: The candidate answer correctly confirms support for two-factor authentication, specifying the use of authenticator apps (which al

### Factual disagreement vs policy disagreement

Read the reasons on both sides and ask which kind of disagreement you have.

- *"the candidate says 90 days, the reference says 30"* — **factual**. One side is wrong; correct it.
- *"the candidate adds a claim about the Retry-After header that the reference never mentions"* — **policy**. Nobody misread anything. Two rules are meeting.

In our strict run, the v1 judge described unrequested extra detail as *"extra useful information"* on exactly the rows the labeller marked *"non verifiable"*. Same text, same observation, opposite verdicts, nineteen times — and not one of them an accuracy failure.

These need opposite responses, and conflating them is how eval programmes stall. **A team that treats a policy disagreement as an accuracy problem will tune the judge prompt forever without converging**, because there is nothing to converge on until somebody decides the rule.

**The diagnostic:** if you showed both parties the same example, would they disagree about what the text *says*, or about what *should count*? If it is the second, stop tuning and go write the policy down.

## Step 5 — What to actually do

**Decide the policy before you build the judge.** It is a product decision, not an engineering one: does an unverifiable-but-plausible claim reaching a customer count as a defect? Answer that with whoever owns the risk, and write it down in a sentence.

**Then encode it in the prompt, and calibrate against labels produced under the same rule.** Now agreement means something — it measures how faithfully the judge implements an agreed policy, which is a question with a right answer.

**Check the error direction, not just the score.** In our run, stating the strict policy took agreement from 55.6% to 84.4%, but the result that mattered more was the failure mode inverting: 19 cases of the judge passing what a human failed became **0**. Every residual error became the judge being *stricter* than the human. For a release gate, fail-safe beats fail-open by more than twenty-nine points of agreement is worth.

**Version the policy, not just the prompt.** The policy document is the primary artifact; the prompt is a rendering of it. When they drift apart the prompt is wrong by definition — a far easier argument to settle than "is this verdict correct?"

**Never sample to zero.** Whatever autonomy the judge earns, keep a small share of production verdicts going to human review permanently. It is the cheapest instrument you have, and it gives you a continuous agreement rate rather than a memory of one.

**Never compare scores across judge versions.** Changing the prompt changes the instrument *and* possibly the policy. Re-baseline instead.

In [11]:
# Disputes against your chosen policy are the highest-value labelled data you produce:
# they mark where the policy is still ambiguous, or where the judge misapplies it.
CHOSEN_POLICY = "strict"      # set to the policy your team actually decided on
CHOSEN_JUDGE = "v3_strict"    # the prompt that encodes it

labels = POLICIES[CHOSEN_POLICY]
disputes = [r for r in examples if labels[r["id"]] != r[CHOSEN_JUDGE]]

fields = ["id", "question", "reference_answer", "candidate_answer", "human", "judge", "judge_reason"]
with Path("calibration_disputes.csv").open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=fields)
    writer.writeheader()
    for row in disputes:
        writer.writerow({
            "id": row["id"], "question": row["question"],
            "reference_answer": row["reference_answer"], "candidate_answer": row["candidate_answer"],
            "human": labels[row["id"]], "judge": row[CHOSEN_JUDGE],
            "judge_reason": row[f"{CHOSEN_JUDGE}_reason"],
        })

print(f"{len(disputes)} disputes between {CHOSEN_JUDGE} and the {CHOSEN_POLICY} labels")
for row in disputes[:4]:
    print(f"  [{row['id']}] human={labels[row['id']]} judge={row[CHOSEN_JUDGE]} :: {row[f'{CHOSEN_JUDGE}_reason'][:110]}")

7 disputes between v3_strict and the strict labels
  [q01] human=PASS judge=FAIL :: Rule 1 broken: The candidate does not explicitly state that the timeout is due to inactivity, which is a key p
  [q14] human=PASS judge=FAIL :: The candidate adds new factual claims not present in the reference: specifically, that Internet Explorer is no
  [q31] human=PASS judge=FAIL :: The candidate adds new factual claims not present in the reference, such as details about exchanging metadata,
  [q32] human=PASS judge=FAIL :: Rule 1 broken: The candidate does not mention that pagination is cursor-based or that the default page size is


Read every dispute. They split into two groups, and both are useful:

- **The judge misapplied the stated policy.** A genuine judge error — the thing calibration is supposed to find.
- **The judge applied the policy more consistently than the human did.** In our strict run this was most of them: rows where the labeller had been lenient about an omission, or had let extra detail through against their own rule.

The second group is not an embarrassment. It means the written policy is now more consistent than the person who wrote it, which is exactly what writing it down was for. The next round either tightens the labelling or adds a clause — *"clarifying detail about a fact already in the reference is acceptable; new facts are not"* — and the disputes get re-run.

This is why disputes are the valuable output: they are your policy's ambiguity surface, found automatically.

## Running this at scale with the Evals API

Everything above runs locally so the mechanics stay visible. Once the judge encodes an agreed policy and you have measured its agreement against labels produced under that same policy, the Evals API gives you the same grading as a managed, repeatable run — which is what you want wired into CI.

The calibration is what makes that grader trustworthy. Wiring up an unvalidated judge automates whatever policy it happened to infer.

In [12]:
eval_obj = client.evals.create(
    name="support-answer-policy-conformance",
    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "question": {"type": "string"},
                "reference_answer": {"type": "string"},
                "candidate_answer": {"type": "string"},
            },
            "required": ["question", "reference_answer", "candidate_answer"],
        },
        "include_sample_schema": False,
    },
    testing_criteria=[
        {
            "type": "label_model",
            "name": "policy-conformance",
            "model": JUDGE_MODEL,
            "input": [{"role": "user", "content": PROMPTS[CHOSEN_JUDGE]}],
            "labels": ["PASS", "FAIL"],
            "passing_labels": ["PASS"],
        }
    ],
)

print(f"created eval {eval_obj.id}")
print("the grader now encodes a written policy — re-calibrate whenever either one changes")

created eval eval_6a7c3bef65988191b4d254ddb948ece4
the grader now encodes a written policy — re-calibrate whenever either one changes


## Takeaways

1. **A judge prompt encodes a policy, not a skill.** Three prompts, one dataset, two labelling rules — and which prompt wins depends entirely on the rule. v3 was best against strict labels (84.4%) and worst against permissive ones (48.9%), from a single cached set of verdicts.
2. **Agreement measures distance from whichever policy someone wrote down.** "Our judge agrees with humans 90% of the time" is an incomplete sentence: agreement with whom, grading under which rule?
3. **The same judge can be broken or good depending only on the labels.** v1 scored 55.6% against a 66.7% constant-predictor baseline under one policy — worse than always answering FAIL — and 86.7% against a 68.9% baseline under the other. Identical verdicts.
4. **You can raise agreement by fixing the judge or by changing your mind about correctness.** The number cannot tell those apart, and only one of them is engineering.
5. **Decide the policy first — it is a product decision.** Does an unverifiable-but-plausible claim reaching a customer count as a defect? Answer that with whoever owns the risk, *before* anyone writes a judge prompt.
6. **Separate factual disagreements from policy disagreements.** The first has a right answer; the second needs a decision. Treating a policy dispute as an accuracy problem is how teams tune prompts forever without converging.
7. **Report κ and a constant-predictor baseline alongside raw agreement.** Under strict labels v1 scored 55.6% with κ = 0.23 — below baseline yet positive κ, the signature of a miscalibrated threshold rather than a useless model. Neither number alone shows that.
8. **Averages hide it, slices find it.** Overall agreement moved a few points between v1 and v2; the long-answer slice was 26.7% vs 100% between the two policies. That slice was the entire story.
9. **Watch the failure mode, not just the score.** Against strict labels, stating the policy took 19 cases of the judge passing what a human failed down to **0** — fail-open to fail-safe. For a release gate that is worth more than the points.
10. **Disputes are your most valuable output.** They mark where the policy is still ambiguous — and in our strict run they showed the written policy had become *more consistent than the human who wrote it*.

### Where to go next

- Measure **human–human agreement** on thirty examples. Two experts disagreeing with each other is policy ambiguity found before a model was ever involved.
- Run **two judges with opposite policies in production** and route the examples where they disagree to human review — that set is your policy's ambiguity surface, generated automatically. Our long-answer slice (100% vs 0%) is what that surface looks like.
- Wire the calibration set into CI so a **judge prompt change re-runs it**, the same way a code change runs tests. A prompt edit is a policy edit; treat it like one.